# 11 — Held-out test set & real-world evaluation

**Purpose:** close the methodological gap flagged in `project-no-held-out-test-set` memory.

Three evaluation points:

1. **Val set** (0.750 macro F1) — what's in `models/runs/v1_2026-05-16/baselines.json`
2. **Real-world 264 manual labels** (0.630 weighted F1) — atlas-ed-data inference output, weeks 1-13
3. **Held-out test** (issues 88-102, never seen in training) — built here

This notebook also covers per-issue F1 (temporal stability), confusion matrix worked errors, and per-source fairness — together they address AM2 criteria K20, S22, S24 + distinction S9/K14.


In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score,
    ConfusionMatrixDisplay,
)
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

MODEL_DIR = ROOT / "models" / "runs" / "v1_2026-05-16"
CLASSIFIER_PATH = MODEL_DIR / "classifier.joblib"
TEST_CSV = ROOT / "data" / "processed" / "test.csv"
TRAIN_CSV = ROOT / "data" / "modelling" / "train.csv"
VAL_CSV = ROOT / "data" / "modelling" / "val.csv"

with open(MODEL_DIR / "run_metadata.json") as f:
    run_meta = json.load(f)
print(f"Model: {run_meta['variant']}")
print(f"Val macro F1 (baseline): {run_meta['metrics']['val_macro_f1']}")
print(f"Real-world weighted F1 (264 labels): {run_meta['metrics']['real_world_weighted_f1']}")


## Step 1 — Extract issues 88+ into a held-out test CSV

In [ ]:
# Only extract if test.csv doesn't already exist (idempotent).
# Issues 1-87 are in train/val; 88-102 are untouched.
# TODO: adapt this to call s00_extract_newsletters.py's parsing functions
# scoped to issues 88+. The existing s00 extracts all issues — you may
# need to add a --since-issue flag or run it then filter.

if TEST_CSV.exists():
    print(f"Using existing {TEST_CSV.name}")
    test_df = pd.read_csv(TEST_CSV)
else:
    raise FileNotFoundError(
        f"Build {TEST_CSV} first. Suggested approach:\n"
        "  1. Run src/training_data/s00_extract_newsletters.py on issues 88+\n"
        "  2. Drop any rows whose `theme` is 'update_from_pi' or 'update_from_programme'\n"
        "  3. Save to data/processed/test.csv with columns: id, newsletter_number,\n"
        "     issue_date, title, theme, organisation, org_broad_category, item_type"
    )

print(f"Held-out test: {len(test_df)} items across issues "
      f"{test_df['newsletter_number'].min()}-{test_df['newsletter_number'].max()}")


In [ ]:
# Sanity check — no URL overlap with train/val.
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
overlap_train = set(test_df.get('link', test_df.get('url', []))) & set(train_df.get('link', train_df.get('url', [])))
overlap_val = set(test_df.get('link', test_df.get('url', []))) & set(val_df.get('link', val_df.get('url', [])))
print(f"URL overlap with train: {len(overlap_train)}  (must be 0)")
print(f"URL overlap with val:   {len(overlap_val)}    (must be 0)")


In [ ]:
# Label distribution test vs train.
print('=== Test label distribution ===')
print(test_df['theme'].value_counts())
print()
print('=== Train label distribution (for comparison) ===')
print(train_df['target'].value_counts() if 'target' in train_df else train_df['theme'].value_counts())


## Step 2 — Run production model on held-out test

In [ ]:
# Load the production model (SBERT no-meta + LogReg).
classifier = joblib.load(CLASSIFIER_PATH)
encoder = SentenceTransformer(run_meta['embedding_model'])
print(f"Loaded classifier + encoder.")

# Title-only input (matches training).
test_titles = test_df['title'].fillna('').astype(str).tolist()
test_embeddings = encoder.encode(test_titles, show_progress_bar=True)
print(f"Encoded {len(test_embeddings)} items, dim={test_embeddings.shape[1]}")


In [ ]:
# Predict.
y_pred = classifier.predict(test_embeddings)
y_pred_proba = classifier.predict_proba(test_embeddings)

# Map curator themes to the 6 model classes if needed.
# Adjust this mapping per your actual label vocabulary.
LABEL_MAP = {
    'teacher_workforce': 'teacher_rrd',
    'teacher_rrd': 'teacher_rrd',
    'edtech': 'edtech',
    'political_environment': 'political_environment_key_organisations',
    'political_environment_key_organisations': 'political_environment_key_organisations',
    'four_nations': 'four_nations',
    'research_practice_policy': 'policy_practice_research',
    'policy_practice_research': 'policy_practice_research',
    'what_matters': 'what_matters_ed',
    'what_matters_ed': 'what_matters_ed',
}
y_true = test_df['theme'].map(LABEL_MAP).fillna(test_df['theme'])
print(f"Unmapped labels (if any): {set(test_df['theme']) - set(LABEL_MAP)}")


## Step 3 — Headline metrics

In [ ]:
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
top1_acc = accuracy_score(y_true, y_pred)

# Top-2 accuracy.
classes = classifier.classes_
top2_idx = np.argsort(-y_pred_proba, axis=1)[:, :2]
top2_preds = classes[top2_idx]
top2_acc = np.mean([yt in tp for yt, tp in zip(y_true, top2_preds)])

print(f"Held-out test (n={len(test_df)})")
print(f"  Macro F1:    {macro_f1:.3f}  (val baseline: {run_meta['metrics']['val_macro_f1']})")
print(f"  Weighted F1: {weighted_f1:.3f}  (real-world 264-label baseline: {run_meta['metrics']['real_world_weighted_f1']})")
print(f"  Top-1 acc:   {top1_acc:.3f}")
print(f"  Top-2 acc:   {top2_acc:.3f}  (val baseline top-2: {run_meta['metrics']['real_world_top2_accuracy']})")


## Step 4 — Per-class breakdown

In [ ]:
print(classification_report(y_true, y_pred, zero_division=0))


## Step 5 — Confusion matrix

In [ ]:
labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(cm, display_labels=labels)
disp.plot(ax=ax, cmap='Blues', values_format='d', xticks_rotation=45)
ax.set_title('Held-out test — confusion matrix')
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'held_out_confusion_matrix.png', dpi=120)
plt.show()


## Step 6 — Worked errors (off-diagonal cells with ≥3 examples)

In [ ]:
# For each off-diagonal (true, pred) pair with ≥3 examples, print sample titles.
test_df['_y_true'] = y_true.values
test_df['_y_pred'] = y_pred
test_df['_correct'] = test_df['_y_true'] == test_df['_y_pred']

errors = test_df[~test_df['_correct']]
pair_counts = errors.groupby(['_y_true', '_y_pred']).size().reset_index(name='n')
pair_counts = pair_counts[pair_counts['n'] >= 3].sort_values('n', ascending=False)

print(f"Off-diagonal pairs with ≥3 errors: {len(pair_counts)}\n")
for _, row in pair_counts.iterrows():
    true_class, pred_class, n = row['_y_true'], row['_y_pred'], row['n']
    print(f"--- {true_class} → predicted {pred_class} ({n} errors) ---")
    sample = errors[(errors['_y_true'] == true_class) & (errors['_y_pred'] == pred_class)].head(3)
    for _, r in sample.iterrows():
        print(f"  '{(r.get('title') or '')[:100]}'")
    print()


## Step 7 — Per-issue F1 (temporal stability)

In [ ]:
# Per-issue macro F1 — does the model hold up over time?
per_issue = []
for issue, g in test_df.groupby('newsletter_number'):
    if len(g) < 5:
        continue
    f1 = f1_score(g['_y_true'], g['_y_pred'], average='macro', zero_division=0)
    per_issue.append({'issue': issue, 'n': len(g), 'macro_f1': f1})

per_issue_df = pd.DataFrame(per_issue).sort_values('issue')
print(per_issue_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(per_issue_df['issue'], per_issue_df['macro_f1'], marker='o')
ax.axhline(run_meta['metrics']['val_macro_f1'], color='grey', linestyle='--', label=f"Val baseline ({run_meta['metrics']['val_macro_f1']})")
ax.set_xlabel('Newsletter issue number')
ax.set_ylabel('Macro F1')
ax.set_title('Held-out per-issue F1 — temporal stability')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'held_out_per_issue_f1.png', dpi=120)
plt.show()


## Conclusion

Three-point evaluation arc:

| Point | Macro F1 / metric | Source |
|---|---|---|
| Val | 0.750 | Single split, used for model selection |
| Real-world 264 labels | 0.630 weighted | Atlas-ed inference output |
| Held-out test (issues 88+) | (computed above) | Fully unseen |

**Honest caveat:** the val set was used for the SHAP-driven model switch + 5+ other decisions, so it's overfit. The held-out test is the first uncontaminated number. Per-issue F1 (plot above) shows temporal stability or drift.

For AM2 distinction (S9/K14): if held-out F1 is close to val, the model is genuinely robust. If it's much lower, that's also a valuable finding — concept drift confirmed, justifies the retraining trigger criteria in [`docs/decisions/model_v1_state_and_retraining_plan.md`](../docs/decisions/model_v1_state_and_retraining_plan.md).